In [159]:
# Import necessary libraries
import pandas as pd
import numpy as np
import requests
import json

In [160]:
# Load in site water level data dictionary
with open('/capstone/aridgw/data/site_data/site_coords.json', 'r') as f:
    site_coords = {k: tuple(v) for k, v in json.load(f).items()}

# Load in clean water level data csv that will be merged to
clean_site_data = "/capstone/aridgw/data/site_data/clean_site_waterdata.csv"
df_sites = pd.read_csv(clean_site_data)

In [164]:

# Use API to access OpenET Data
header = {"Authorization": "31WmFjhG0zzzyqT7LyXfuTS9sHopnzU90kxnLRFSPAuwE8w4N7kg6zuPeBRs"}

# Set sites query parameter to site coordinates dictionary
sites = site_coords

# Select boundary box side length
size_km = 1

# Extract data sections less than 10 years as required by the API
date_ranges = [
    ("2000-01-01", "2009-12-31", 2.0), 
    ("2010-01-01", "2019-12-31", 2.0),
    ("2020-01-01", "2025-12-31", 2.1), # Use version 2.1 for newer versions which may not have accurate data in older versions
]

dfs = [] # Create empty dataframe

# Create function to generate boundary box given coordinates and boundary box size
def make_square_geometry(lat, lon, size_km=size_km):
    half = size_km / 2
    lat_offset = half / 111.0
    lon_offset = half / (111.0 * np.cos(np.radians(lat)))
    return [
        lon - lon_offset, lat + lat_offset,
        lon - lon_offset, lat - lat_offset,
        lon + lon_offset, lat - lat_offset,
        lon + lon_offset, lat + lat_offset
    ]

# Use API to extract ET data 
for name, (lat, lon) in sites.items():
    print(f"Querying {name}...")
    
    for start_date, end_date, version in date_ranges:

        polygon_args = { # Define query paramters
            "date_range": [start_date, end_date],
            "interval": "monthly",
            "geometry": make_square_geometry(lat, lon, size_km),
            "model": "Ensemble",
            "reducer": "mean",
            "variable": "ET",
            "reference_et": "gridMET",
            "units": "mm",
            "file_format": "JSON",
            "version": version
        }

        resp = requests.post( # Request polygon API for this extraction
            headers=header,
            json=polygon_args,
            url="https://openet-api.org/raster/timeseries/polygon"
        )

        print(f"  {name} {start_date}–{end_date} — Status: {resp.status_code}") # Report status of data to console

        if resp.status_code != 200: # Return error if unable to extract data
            print(f"  ERROR: {resp.json()}")
            continue

        data = resp.json()
        df = pd.DataFrame(data)

        # Add columns to dataframe to specify site specific information
        df["site"] = name
        df["latitude"] = lat
        df["longitude"] = lon
        df["bbox_side"] = size_km
        df["open_et_version"] = version
        dfs.append(df)

df_monthly = pd.concat(dfs, ignore_index=True) # Add new data as new observations in dataframe
print(df_monthly.head())

Querying KSGS.371852100505801...
  KSGS.371852100505801 2000-01-01–2009-12-31 — Status: 200


KeyboardInterrupt: 

In [152]:
# Delete later raw monthly et
df_monthly = "/Users/richardmonteslemus/capstone/GW-depth-data-exploration/openet_data_monthly_clean.csv"
df_monthly = pd.read_csv(df_monthly)

In [153]:
df_monthly

,time,et,site,latitude,longitude,bbox_side,open_et_version,year
0,1/1/00,12.573,KSGS.371852100505801,37.31502,-100.8505,1,2.0,2000
1,2/1/00,27.326,KSGS.371852100505801,37.31502,-100.8505,1,2.0,2000
2,3/1/00,40.734,KSGS.371852100505801,37.31502,-100.8505,1,2.0,2000
3,4/1/00,55.314,KSGS.371852100505801,37.31502,-100.8505,1,2.0,2000
4,5/1/00,104.492,KSGS.371852100505801,37.31502,-100.8505,1,2.0,2000
...,...,...,...,...,...,...,...,...
15595,8/1/25,70.545,southern_willcox,31.36597,-109.6630,1,2.1,2025
15596,9/1/25,44.656,southern_willcox,31.36597,-109.6630,1,2.1,2025
15597,10/1/25,47.854,southern_willcox,31.36597,-109.6630,1,2.1,2025
15598,11/1/25,27.207,southern_willcox,31.36597,-109.6630,1,2.1,2025


In [154]:
# Convert time column to datetime, extract year
df_monthly["time"] = pd.to_datetime(df_monthly["time"], format="%m/%d/%y")
df_monthly["year"] = df_monthly["time"].dt.year
df_monthly = df_monthly.rename(columns={"et": "monthly_et_avg"})
df_monthly

,time,monthly_et_avg,site,latitude,longitude,bbox_side,open_et_version,year
0,2000-01-01,12.573,KSGS.371852100505801,37.31502,-100.8505,1,2.0,2000
1,2000-02-01,27.326,KSGS.371852100505801,37.31502,-100.8505,1,2.0,2000
2,2000-03-01,40.734,KSGS.371852100505801,37.31502,-100.8505,1,2.0,2000
3,2000-04-01,55.314,KSGS.371852100505801,37.31502,-100.8505,1,2.0,2000
4,2000-05-01,104.492,KSGS.371852100505801,37.31502,-100.8505,1,2.0,2000
...,...,...,...,...,...,...,...,...
15595,2025-08-01,70.545,southern_willcox,31.36597,-109.6630,1,2.1,2025
15596,2025-09-01,44.656,southern_willcox,31.36597,-109.6630,1,2.1,2025
15597,2025-10-01,47.854,southern_willcox,31.36597,-109.6630,1,2.1,2025
15598,2025-11-01,27.207,southern_willcox,31.36597,-109.6630,1,2.1,2025


In [117]:
# Convert monthly ET data into csv
df_monthly.to_csv(f"open_et_data/openet_data_monthly_{size_km}km.csv", index=False)

In [118]:
# Check number of monthly et observations per site
# Ensure value consistency
df_monthly.groupby(["site"]).size().reset_index(name="n") 

,site,n
0,KSGS.371852100505801,312
1,KSGS.372043101363101,312
2,KSGS.372539100142504,312
3,KSGS.373331098033301,312
4,KSGS.373607100565301,312
5,KSGS.374111099070401,312
6,KSGS.374125100344101,312
7,KSGS.374747100552101,312
8,KSGS.375145100485701,312
9,KSGS.375454101075401,312


In [ ]:
# Calculate a single monthly average per year 
df_annual = (
    df_monthly
    .groupby(["site", "year", "latitude", "longitude", "bbox_side", "open_et_version"])['monthly_et_avg']
    .mean()
    .reset_index()
    .rename(columns={"monthly_et_avg": "annual_et_avg"})
)

In [ ]:
# Rescale annual et 
df_annual["scaled_annual_et_avg"] = (df_annual["mean_annual_et"] * 12).round(3)
df_annual = df_annual.drop(columns=["annual_et_avg"])


KeyError: 'annual_et_avg'

In [53]:
# Check number of scaled_annual_et_avg values per year
# Ensure value consistency
df_annual.groupby(["year"]).size().reset_index(name="n") 

,year,n
0,2000,50
1,2001,50
2,2002,50
3,2003,50
4,2004,50
5,2005,50
6,2006,50
7,2007,50
8,2008,50
9,2009,50


In [42]:
# Convert annual ET data into csv
df_annual.to_csv(f"open_et_data/openet_data_annual_{size_km}km.csv", index=False)

In [55]:
# Remove lat and long to above columns duplicates
df_annual = df_annual.drop(columns=["latitude", "longitude"])

KeyError: "['latitude', 'longitude'] not found in axis"

In [56]:
df_annual

,site,year,bbox_side,version,scaled_annual_et_avg
0,KSGS.371852100505801,2000,1,2.0,781.134
1,KSGS.371852100505801,2001,1,2.0,859.635
2,KSGS.371852100505801,2002,1,2.0,787.151
3,KSGS.371852100505801,2003,1,2.0,836.748
4,KSGS.371852100505801,2004,1,2.0,666.882
...,...,...,...,...,...
1295,willcox,2021,1,2.1,611.347
1296,willcox,2022,1,2.1,799.466
1297,willcox,2023,1,2.1,861.585
1298,willcox,2024,1,2.1,825.554


In [57]:
# Load in clean water level data csv that will be merged to
# et_annual_data = "/capstone/aridgw/data/site_data/openet_et_annual_mean.csv"
# df_et_annual = pd.read_csv(et_annual_data)

# Load in clean water level data csv that will be merged to
clean_site_data = "/capstone/aridgw/data/site_data/clean_site_waterdata.csv"
df_sites = pd.read_csv(clean_site_data)

In [58]:
df_sites.groupby(["site_id"]).size().reset_index(name="n") 

,site_id,n
0,KSGS.371852100505801,21
1,KSGS.372043101363101,21
2,KSGS.372539100142504,21
3,KSGS.373331098033301,21
4,KSGS.373607100565301,21
5,KSGS.374111099070401,21
6,KSGS.374125100344101,21
7,KSGS.374747100552101,21
8,KSGS.375145100485701,21
9,KSGS.375454101075401,21


In [61]:
# Merge ET data to water level data 
df_merged_et = pd.merge(
    df_sites,
    df_annual,
    left_on=['site_id', 'year_value'],
    right_on=['site', 'year'],
    how='outer'
)

In [64]:
# Remove rows containing NA in columns important for analysis
df_merged_et = df_merged_et.dropna(subset=["depth_to_gw_m", "scaled_annual_et_avg"])

In [65]:
# Check number of observations per site in merged df
df_merged_et.groupby(["site_id"]).size().reset_index(name="n")

,site_id,n
0,KSGS.371852100505801,21
1,KSGS.372043101363101,21
2,KSGS.372539100142504,21
3,KSGS.373331098033301,21
4,KSGS.373607100565301,21
5,KSGS.374111099070401,21
6,KSGS.374125100344101,21
7,KSGS.374747100552101,21
8,KSGS.375145100485701,21
9,KSGS.375454101075401,21


In [ ]:
# Remove duplicate year column
df_merged_et = df_merged_et.drop(columns='year')
df_merged_et = df_merged_et.drop(columns='site')

In [137]:
df_merged_manual

,year_value,site_id,depth_to_gw_ft,depth_to_gw_m,year_type_id,metric_id,Aq_type,trend_type,POI_type_ID,latitude,longitude,location_name,data_source,bbox_side,open_et_version,scaled_annual_et_avg
0,2000,KSGS.371852100505801,239.390000,72.966072,water,mean,unconfined,annual mean,2000.0,37.31502,-100.8505,NaN,USGS,1,2.0,781.134
1,2001,KSGS.371852100505801,241.960000,73.749408,water,mean,unconfined,annual mean,2000.0,37.31502,-100.8505,NaN,USGS,1,2.0,859.635
2,2002,KSGS.371852100505801,242.780000,73.999344,water,mean,unconfined,annual mean,2000.0,37.31502,-100.8505,NaN,USGS,1,2.0,787.151
3,2003,KSGS.371852100505801,246.710000,75.197208,water,mean,unconfined,annual mean,2000.0,37.31502,-100.8505,NaN,USGS,1,2.0,836.748
4,2004,KSGS.371852100505801,247.710000,75.502008,water,mean,unconfined,annual mean,2000.0,37.31502,-100.8505,NaN,USGS,1,2.0,666.882
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1006,2020,willcox,402.899947,122.803900,NaN,NaN,NaN,NaN,NaN,32.03619,-109.7540,Arizona (Willcox Basin),Jasechko,1,2.1,703.334
1007,2021,willcox,410.200144,125.029000,NaN,NaN,NaN,NaN,NaN,32.03619,-109.7540,Arizona (Willcox Basin),Jasechko,1,2.1,611.347
1008,2022,willcox,432.200145,131.734600,NaN,NaN,NaN,NaN,NaN,32.03619,-109.7540,Arizona (Willcox Basin),Jasechko,1,2.1,799.466
1009,2023,willcox,444.500014,135.483600,NaN,NaN,NaN,NaN,NaN,32.03619,-109.7540,Arizona (Willcox Basin),Jasechko,1,2.1,861.585


In [96]:
df_merged_et.equals(df_merged_manual)

False

In [ ]:
# Turn merged df into csv
df_merged_et.to_csv("merged_openet_data.csv", index=False)